Groundwater | Transport Modeling Track

# Step 3: From Perceptual to Conceptual — MODFLOW 6 for Solute Transport

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → **3-MODFLOW** → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 2, you built a perceptual model of **contaminant transport** through the Limmat Valley aquifer — a contaminant that **spills upgradient of a groundwater heat-exchange (GWHE) doublet** and is carried by the doublet's flow field toward the **extraction well**, which doubles as the monitoring / compliance well. You estimated a domain-average seepage velocity of ~2.3 m/d, set an effective porosity of $n_e = 0.20$ and a longitudinal dispersivity $\alpha_L = 10$ m, and framed the operational question: **does the plume reach the extraction well above the regulatory threshold, and when — or bypass it?** Now we translate these insights into the language of MODFLOW 6: the transport equation (with retardation and decay), numerical accuracy criteria, and the GWT solute-transport packages.

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~25–35 min core + 10 min optional sections |
| **Prerequisites** | Flow track complete, Transport Steps 1–2 |

**Learning objectives:**

By the end of this notebook you will be able to:

1. Write the **advection-dispersion-reaction equation (ADRE)** for a dissolved solute — including retardation $R$ and first-order decay $\lambda$ — and explain each term
2. Evaluate **grid Peclet and Courant numbers** as transport *accuracy* criteria
3. Identify **MODFLOW 6 GWT packages**, and choose the right **source package** (SRC vs CNC vs WEL+SSM) for a spill
4. Map **perceptual transport features** to specific GWT package inputs

In [ ]:
# Setup
import sys

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from shared_functions import check_task_with_solution, create_multiple_choice

---

## Introduction

In the flow track (Step 3), you learned how MODFLOW 6 discretises the **flow equation** — control volumes, hydraulic conductivity averaging, grid types, and boundary condition packages. All of that carries over unchanged to a transport simulation: **the same grid, the same flow solution, the same boundary conditions drive transport.**

What's new for transport?

| Already covered (flow NB3) | New in this notebook |
|---|---|
| Governing flow equation | Advection-dispersion equation (ADE) |
| CVFD discretisation, K-averaging | Numerical stability (Peclet, Courant) |
| Grid types (DIS, DISV, DISU) | Transport-specific packages (GWT) |
| GWF packages (RIV, RCH, WEL, …) | How GWF fluxes carry solute concentration (SSM) |

> 💡 **Why MODFLOW 6 for transport?** The Groundwater Transport (GWT) model is fully integrated into MODFLOW 6. It shares the same grid as the Groundwater Flow (GWF) model, runs in the same simulation, and is fully scriptable via FloPy — no separate transport code needed. (A sister model, GWE, does the same for heat; see the optional box at the end of this notebook.)

## 1 — The Advection-Dispersion(-Reaction) Equation

### Governing equation

The **advection-dispersion-reaction equation (ADRE)** describes how a dissolved solute is transported and transformed in a porous medium:

$$\frac{\partial (n R\, C)}{\partial t} = \nabla \cdot (n \mathbf{D} \nabla C) - \nabla \cdot (n \mathbf{v} C) - \lambda\, n R\, C + q_s$$

| Term | Name | Physical meaning |
|---|---|---|
| $\partial(nRC)/\partial t$ | **Retarded storage** | Change in stored mass; the **retardation factor** $R = 1 + \rho_b K_d / n$ accounts for reversible sorption ($R=1$ = none) |
| $\nabla \cdot (n \mathbf{D} \nabla C)$ | **Dispersion** | Spreading due to velocity variations + molecular diffusion |
| $\nabla \cdot (n \mathbf{v} C)$ | **Advection** | Bulk movement with the flowing groundwater |
| $\lambda\, n R\, C$ | **First-order decay** | Mass loss ∝ concentration (biodegradation, radioactive decay); $\lambda = \ln 2 / t_{1/2}$ |
| $q_s$ | **Sources / sinks** | Mass added or removed (the spill source, wells, rivers) |

where $n$ is porosity, $C$ concentration, $\mathbf{v}$ the seepage velocity, $\mathbf{D}$ the hydrodynamic dispersion tensor, $R$ the retardation factor, and $\lambda$ the first-order decay constant. The **conservative tracer** is the special case $R = 1,\ \lambda = 0$ — the simplest, cleanest starting point — and a **sorbing** ($R>1$) or **decaying** ($\lambda>0$) contaminant is one incremental step on top (the **MST** package switches each on; see §3).

### Flow vs. transport: a comparison

| | Flow equation | Transport equation |
|---|---|---|
| **State variable** | Hydraulic head $h$ | Concentration $C$ |
| **What drives movement** | Hydraulic gradient $\nabla h$ | Groundwater velocity $\mathbf{v}$ (advection) + gradient $\nabla C$ (dispersion) |
| **What spreads it** | Hydraulic conductivity $K$ | Dispersion tensor $\mathbf{D}$ (= $\alpha_L v$ + molecular diffusion $D_m^*$) |
| **Storage** | Specific storage $S_s$ | Effective porosity $n$ × retardation $R$ |
| **Coupling** | Independent | **One-directional:** flow → transport (velocity field) |

> 💡 **Transport is coupled but one-directional — and it *does* depend on $K$.** GWT doesn't take $K$ as a direct input, but it runs entirely on the velocity field GWF computed *from* $K$ ($\mathbf{v} = -K\nabla h / n$), so $K$ uncertainty propagates straight into transport. For a dilute solute the concentration doesn't change the water density, so there's no feedback the other way. (Density-dependent flow — brines, strong thermal gradients — is a separate topic; it would need a variable-density code such as MODFLOW 6's **BUY** package, SEAWAT, or SUTRA.)

### A note on analytical verification

The theory lectures solve simplified forms of the ADE analytically. These are **verification toys** — small, uniform-flow problems with a known answer against which to check the numerical scheme, *not* benchmarks for the doublet itself:

- **1D transient (Ogata–Banks)**, a continuous constant-concentration inlet in 1D uniform flow, verifies **arrival timing** (toe, $t_{50}$, front sharpness):
$$\frac{C}{C_0} = \tfrac{1}{2}\,\mathrm{erfc}\!\left[\frac{x - vt}{2\sqrt{D_L t}}\right] + \tfrac{1}{2}\,\exp\!\left(\frac{vx}{D_L}\right)\mathrm{erfc}\!\left[\frac{x + vt}{2\sqrt{D_L t}}\right]$$
(keep the full two-term form; the single-term simplification is only accurate at large Peclet).
- **2D mass-loading** ($\dot{M}$ point/line source in 2D uniform flow) carries **transverse dispersion** $D_T$, so it's the tool for checking **plume shape / lateral width** — introduced later in the track.

> ⚠️ **These are idealisations, not the doublet.** Both assume **uniform** flow; the doublet's convergent, non-uniform field will *not* match them cleanly, so a clean match is reserved for a dedicated **1D uniform-flow verification toy** (Step 8). Treat a departure as a limitation of the analytical idealisation, not a model failure.

In [ ]:
# Checkpoint 1 — ADE Comprehension
create_multiple_choice('task_t03_checkpoint_1')

---

## 2 — Numerical Stability: Peclet and Courant Numbers

Transport is numerically **harder** than flow. The flow equation is purely diffusive (elliptic), while the ADE includes advection (hyperbolic character) — this makes it prone to oscillations and numerical dispersion if the grid or time step is too coarse.

Two dimensionless numbers control stability:

### Grid Peclet number

$$Pe_{grid} = \frac{v \cdot \Delta x}{D_L} \leq 2$$

This is a **cell-scale** requirement. It says: the advective transport across one cell must not exceed twice the dispersive spreading. Rearranging for maximum cell size:

$$\Delta x \leq \frac{2 \cdot D_L}{v} = 2 \cdot \alpha_L$$

since $D_L = \alpha_L \cdot v$ when molecular diffusion is negligible.

### Courant number

$$Cr = \frac{v \cdot \Delta t}{\Delta x} \leq 1$$

This is a **time-step** requirement. It says: a parcel of solute should not cross more than one cell per time step.

### Worked example with Limmat Valley values

Using domain-average values from NB2: $v \approx 2.3$ m/d, $\alpha_L = 10$ m, so $D_L = \alpha_L \cdot v \approx 23$ m$^2$/d.

| Criterion | Formula | Result |
|---|---|---|
| Max cell size | $\Delta x \leq 2 \alpha_L$ | $\leq 20$ m |
| Max time step (for $\Delta x = 20$ m) | $\Delta t \leq \Delta x / v$ | $\leq 8.7$ days |

Near the doublet wells, local velocities can reach ~7 m/d (the flow converges toward the abstraction well), reducing the Courant limit to $\leq 2.9$ days for the same cell size. With monthly stress periods ($\Delta t = 30$ days) and typical cell sizes of 25–50 m, Courant numbers can exceed 1 in the fast-flow zone — this motivates using the **TVD scheme** (see below).

Compare this to the flow model, where cell sizes of 50–100 m and time steps of days to weeks are common. **Transport demands finer spatial and temporal resolution.**

> ℹ️ **A note on units.** Molecular diffusion in water is $D_m^* \approx 1\times10^{-9}$ m²/s. Because the model runs in **days**, convert before use: $D_m^* \approx 8.64\times10^{-5}$ m²/d (the m/s → m/d factor is ×86 400). At $\alpha_L = 10$ m the mechanical term $\alpha_L v$ dominates, so diffusion is negligible here — but the unit conversion still matters when you enter `diffc` in the DSP package.

### TVD schemes and relaxation

The **TVD (Total Variation Diminishing)** advection scheme available in MODFLOW 6 is less sensitive to the grid Peclet constraint. It can handle $Pe_{grid} > 2$ without spurious oscillations, at the cost of introducing some numerical dispersion. In practice, TVD allows coarser grids than the classical $Pe \leq 2$ rule — but the Courant criterion still applies.

In [ ]:
# Checkpoint 2 — Peclet Calculation
check_task_with_solution('task_t03_checkpoint_2')

## 3 — MODFLOW 6 GWT Packages

MODFLOW 6 implements solute transport through the **Groundwater Transport (GWT)** model. Like GWF, GWT is built from interchangeable packages — each handles one physical process or boundary type.

| Package | Purpose | Spill-study use |
|---|---|---|
| **MST** | Mass storage and transfer: effective porosity $n_e$, optional sorption $K_d$ (retardation $R$), 1st-order decay $\lambda$ | $n_e = 0.20$; conservative core, with $R$/$\lambda$ as opt-in add-ons |
| **ADV** | Advection scheme selection (upstream, central, TVD) | `scheme='TVD'` for accuracy on the doublet grid |
| **DSP** | Dispersion: $\alpha_L$ (`alh`), $\alpha_T$ (`ath1`), molecular diffusion $D_m^*$ (`diffc`) | $\alpha_L = 10$ m, $\alpha_T = 1$ m, $D_m^* = 8.64\times10^{-5}$ m²/d |
| **SRC** | Mass source/loading: per-cell mass-loading **rate** [M/T] | **The spill:** a known contaminant mass released with no water flux |
| **SSM** | Source-sink mixing: concentration carried by each GWF flux | RIV / RCH inflow concentrations (and a water-carrying injection source, Step 8) |
| **CNC** | Constant concentration (Dirichlet BC) | Only a genuine held-concentration source (a dissolving NAPL pool) |
| **IC** | Initial concentration distribution | Background ≈ 0 (clean aquifer) |
| **OC** | Output control (what to save, when) | Concentration fields, **mass budgets** |

### How GWT couples to GWF

The GWT model receives the velocity field from GWF through either a **GWF-GWT Exchange** (both models in one simulation) or the **FMI** (Flow Model Interface, reading a saved GWF solution). Either way, **GWT never solves the flow equation** — it solves the transport equation on velocities GWF computed. Here the flow model is **steady-state**, so transport (transient) runs on a *fixed* velocity field.

### Specifying the source: SRC vs. CNC vs. WEL+SSM

How you introduce the contaminant matters, and the three options are **not** interchangeable:

| Approach | What you control | What the model does | When to use |
|---|---|---|---|
| **SRC** (mass loading) | A mass-loading **rate** [M/T] directly | Adds that mass to the source cells, independent of any water flux — **controlled, grid-independent** | **Our spill:** a known contaminant *mass* released with no associated water flux — **preferred** |
| **CNC** (constant concentration) | A fixed concentration in the cell | **Mass flux floats:** the model supplies whatever mass holds $C$ fixed → grid- and flow-dependent | A true persistent / solubility-controlled source (a NAPL pool dissolving at solubility) |
| **WEL + SSM (AUX concentration)** | Injected concentration $c_{inj}$ on a *pumping* well | Mass rate $= Q_{inj}\cdot c_{inj}$ — controlled, grid-independent, but **adds water** | A *water-carrying* source — the injection-well recirculation study (Step 8) |

> ⚠️ **The CNC pitfall.** CNC fixes the *concentration* and lets the *mass flux* adjust — so the mass entering the aquifer depends on the grid and local flow, and can be far larger than intended. For a **known-mass spill that adds no water**, **SRC** is the honest choice: you set the released mass directly, it adds no water (the doublet flow field is undisturbed), and it is **grid-independent** (which also keeps a native-vs-refined grid comparison clean). CNC is right only when the source really is held at a concentration.

### Common misconceptions

| Misconception | Reality |
|---|---|
| "Transport is independent of hydraulic conductivity $K$" | No — GWT doesn't take $K$ directly, but it runs on the velocity field GWF computed *from* $K$ ($\mathbf{v}=-K\nabla h/n$), so $K$ uncertainty propagates into transport. |
| "Porosity is inherited from GWF" | No — transport uses **effective (kinematic) porosity $n_e$**, set in **MST**. A steady flow model uses *no* porosity; a transient one stores with specific storage / yield — different quantities from $n_e$. |
| "A known-mass spill is a fixed-concentration boundary" | No — a spill of known *mass* is a **mass loading (SRC)**; fixing concentration is CNC, with the floating-mass pitfall above. |
| "Finer time steps are always better" | The Courant criterion sets a *maximum* $\Delta t$ for accuracy; below it, smaller steps only waste computation. |
| "RIV captures all river–aquifer solute exchange" | No — SSM only carries solute on the **advective** water flux; no diffusive exchange at zero net flow. |

<details>
<summary><strong>Optional: GWE (heat) background — the same machinery for temperature</strong> (click to expand)</summary>

The doublet wells in this study are physically a **groundwater heat-exchange (GWHE)** installation: paired injection–abstraction wells that move thermal energy for heating and cooling. Here we study the **conservative solute** that travels with the water, but the *same* MODFLOW 6 transport machinery can model the **heat** directly through the **Groundwater Energy (GWE)** model. You will not build a GWE model in this track — this box is background so you recognise it.

### Heat transport form of the ADE

For heat, concentration $C$ becomes temperature $T$, and energy is stored in **both** the water and the solid grains:

$$\underbrace{\frac{\partial}{\partial t}\bigl[n \rho_w c_w T + (1-n) \rho_s c_s T\bigr]}_{\text{bulk thermal storage}} = \underbrace{\nabla \cdot (\lambda_{bulk} \nabla T)}_{\text{conduction}} + \underbrace{\nabla \cdot (n \rho_w c_w \mathbf{D}_{mech} \nabla T)}_{\text{mechanical dispersion}} - \underbrace{\nabla \cdot (n \rho_w c_w \mathbf{v} T)}_{\text{advection}} + q_E$$

Because heat is stored in both phases, it experiences **thermal retardation** ($R \approx 3.2$ for the Limmat Valley gravels). A conservative solute is stored only in the water, so it is **not** retarded — the clean, simplest case, which is why we use it as the core path.

### GWE ↔ GWT package map

The two model types are structurally almost identical; only the "diffusive" term and the storage term differ:

| GWT package (solute) | GWE package (heat) | What changes |
|---|---|---|
| **MST** (Mass Storage and Transfer) | **EST** (Energy Storage and Transfer) | Sorption $K_d$, decay → heat capacity & density (thermal retardation) |
| **DSP** (Dispersion) | **CND** (Conduction-Dispersion) | Molecular diffusion $D_m^*$ → thermal conductivity $\lambda_{bulk}$ |
| **CNC** (Constant Concentration) | **CTP** (Constant Temperature) | Concentration value → temperature value |
| **SRC** (Mass Source/Loading) | **ESL** (Energy Source/Loading) | Mass rate [M/T] → energy rate [E/T] |
| **ADV** (Advection) | **ADV** | Identical — same scheme, same velocity field |
| **SSM** (Source-Sink Mixing) | **SSM** | Identical — associates concentration / temperature with each GWF stress |
| **IC** (Initial Conditions) | **IC** | Concentration → temperature |
| **OC** (Output Control) | **OC** | Identical |

**Key insight:** mechanical dispersion ($\alpha_L$, $\alpha_T$) is identical in GWT and GWE — it describes velocity-driven spreading that does not depend on *what* is transported. Only the molecular term differs (diffusion $D_m^*$ for solutes, conduction $\lambda_{bulk}$ for heat), and heat adds dual-phase storage (retardation). This is why dispersivity learned from one transfers to the other.

</details>

## 4 — Translating the Perceptual Model to GWT Packages

In NB2, we identified the key transport features of the spill→capture study. Here is how each maps to a MODFLOW 6 GWT package:

| Perceptual feature (from NB2) | Value | GWT package | How it enters the model |
|---|---|---|---|
| Effective porosity $n_e$ | 0.20 | **MST** | `porosity` input; controls pore velocity (and retarded storage) — must equal $n_e$ |
| Longitudinal dispersivity $\alpha_L$ | 10 m | **DSP** | `alh` — longitudinal mechanical dispersion |
| Transverse dispersivity $\alpha_T$ | 1 m | **DSP** | `ath1` — transverse dispersion (single layer) |
| Molecular diffusion $D_m^*$ | $8.64\times10^{-5}$ m²/d | **DSP** | `diffc` — negligible here, but specified for completeness |
| Background concentration | ≈ 0 | **IC** | Initial concentration field (clean aquifer) |
| The spill: released mass $M$, location, duration | source strength | **SRC** | Mass-loading rate [M/T] on the source cells — grid-independent, adds no water |
| River / recharge inflow concentration | scenario-dependent | **SSM** | Auxiliary concentration on RIV / RCH packages |
| Advection scheme | TVD | **ADV** | `scheme='TVD'` for accuracy |
| (Optional) sorption $K_d$ / decay $\lambda$ | conservative core | **MST** | `sorption` / `distcoef` (retardation $R$) and `first_order_decay` / `decay` — opt-in |

> 🤔 **Reflection:** The operational question is **capture vs. bypass** — does the upgradient spill reach the extraction (monitoring) well above the regulatory threshold, and when? In the model, the **SRC** package handles the spill: you specify the *mass-loading rate* on the source cells, so the released mass is exactly what you set — grid-independent and adding no water, so the doublet's flow field stays undisturbed. (The *recirculation* variant — tracer injected *with* the reinjected water — is a water-carrying source and uses WEL+SSM instead; that's a Step 8 application.)

In [ ]:
# Checkpoint 3 — Package Identification
create_multiple_choice('task_t03_checkpoint_3')

## Summary

In this notebook, you learned:

- The **advection-dispersion-reaction equation** extends the flow equation to transport: advection moves the front, dispersion spreads it, sources/sinks add or remove mass, and **retardation $R$** and **first-order decay $\lambda$** capture reactions — a **conservative tracer** ($R=1,\lambda=0$) is the clean core case
- **Numerical accuracy** requires grid Peclet $\leq 2$ (max cell size $= 2\alpha_L$) and Courant $\leq 1$ (max time step $= \Delta x / v$) — transport needs finer grids and shorter time steps than flow; TVD relaxes the Peclet constraint
- Analytical **verification toys** (1D transient Ogata–Banks for arrival; a 2D mass-loading solution for transverse width) check the numerical scheme on uniform-flow problems — they are *not* benchmarks for the convergent doublet flow
- MODFLOW 6 **GWT packages** (MST, ADV, DSP, SRC, SSM, CNC, IC, OC) each handle one aspect of transport; the **known-mass spill** is set with **SRC** (grid-independent, adds no water), *not* CNC (floating mass) — WEL+SSM is reserved for a water-carrying source
- The perceptual model from NB2 **maps directly** to GWT inputs: $n_e \rightarrow$ MST; $\alpha_L$, $\alpha_T \rightarrow$ DSP (`alh`, `ath1`); the **spill $\rightarrow$ SRC**; background $\rightarrow$ IC

### What You're Taking Forward

| Concept | You'll use it in… |
|---|---|
| ADRE / Peclet / Courant criteria | NB4: grid and time-step diagnosis for the transport model |
| GWT packages (MST, DSP, ADV, SRC) | NB4: FloPy implementation of each package |
| Perceptual → GWT package mapping | NB4: parameter assignment from NB2 values |
| SRC spill source vs. CNC pitfall | NB4: setting the spill mass loading correctly |

### Documentation Resources

Bookmark these — you'll need them in NB4:

- **MODFLOW 6 GWT documentation:** [USGS MODFLOW 6 Description of Input and Output](https://modflow6.readthedocs.io/en/latest/)
- **FloPy documentation:** [FloPy — Python interface to MODFLOW](https://flopy.readthedocs.io/en/latest/)

---

## Next Steps

You now have the conceptual framework for solute-transport modelling in MODFLOW 6. In the next step, **04t builds the GWT model** — you implement each package in FloPy on the inherited flow grid.

**Continue to:** [Step 4 — Model Implementation](./04t_model_implementation.ipynb) — Build the GWT model in FloPy

**Review if needed:** [Step 2 — Perceptual Model](./02t_perceptual_model.ipynb) — Transport parameters and source concentrations